In [1]:
import os
import load_dotenv
from load_dotenv import load_dotenv

# This function will load all the variable from the .env file and 
# make them available in the os.environ dictionary (env variables)
load_dotenv()

if os.environ.get("CLAUDE_API_KEY"):
    print("API KEY variable has been loaded")
else:
    raise ValueError("CLAUDE API KEY not found")

API KEY variable has been loaded


#### Parallel Chains

In [ ]:
# TASK - 1 [Prompt]
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate

prompt_template = ChatPromptTemplate.from_messages([
    ("system", "You are a movie summarizer"),
    ("human", "Please summarize the movie in brief: {input}")
])

In [3]:
# TASK - 2 [LLM]
from langchain_anthropic import ChatAnthropic

llm_anthropic = ChatAnthropic(
    model=os.environ.get("CLAUDE_MODEL"), 
    temperature=0,
    api_key=os.environ.get("CLAUDE_API_KEY"),
    base_url=os.environ.get("CLAUDE_BASE_URL")
)
llm_anthropic

ChatAnthropic(metadata={'lc_versions': {'langchain-core': '1.5.3', 'langchain': '1.3.14', 'langchain-anthropic': '1.5.3'}}, output_version=None, model='vertex_ai.anthropic.claude-opus-4-6', max_tokens=4096, temperature=0.0, anthropic_api_url='https://genai-sharedservice-americas.pwcinternal.com', anthropic_api_key=SecretStr('**********'), anthropic_proxy=None, model_kwargs={})

In [4]:
# TASK - 3 [String Parser]
from langchain_core.output_parsers import StrOutputParser

str_parser = StrOutputParser()

In [5]:
# TASK - 4 [Custom Runnable]
from langchain_core.runnables import RunnableLambda

def dictionary_maker(text:str)-> dict:

    return {"text" : text}

dictionary_maker_runnable = RunnableLambda(dictionary_maker)

#### Parallel Chain 1

In [7]:
# TASK - 1 [Prompt]

linkedin_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a LinkedIn post generator"),
    ("human", "Create a post for the following text for LinkedIn: {input}")
])

# TASK - 2 [LLM]
from langchain_anthropic import ChatAnthropic

llm_anthropic = ChatAnthropic(
    model=os.environ.get("CLAUDE_MODEL"), 
    temperature=0,
    api_key=os.environ.get("CLAUDE_API_KEY"),
    base_url=os.environ.get("CLAUDE_BASE_URL")
)
llm_anthropic

# TASK - 3 [String Parser]
from langchain_core.output_parsers import StrOutputParser

str_parser = StrOutputParser()

chain_linkedIn = linkedin_prompt | llm_anthropic | str_parser

#### Parallel Chain 2

In [8]:
def insta_chain(text:dict):

    text = text["text"]

    # TASK - 1 [Prompt]

    insta_prompt = ChatPromptTemplate.from_messages([
        ("system", "You are an Instagram post generator"),
        ("human", "Create a post for the following text for LinkedIn: {text}")
    ])

    # TASK - 2 [LLM]

    llm_anthropic = ChatAnthropic(
        model=os.environ.get("CLAUDE_MODEL"), 
        temperature=0,
        api_key=os.environ.get("CLAUDE_API_KEY"),
        base_url=os.environ.get("CLAUDE_BASE_URL")
    )
    llm_anthropic

    # TASK - 3 [String Parser]

    str_parser = StrOutputParser()

    chain_insta = insta_prompt | llm_anthropic | str_parser

    result = chain_insta.invoke(text)

    return result


insta_chain_runnable = RunnableLambda(insta_chain)

#### Final Orchestration

In [ ]:
from langchain_core.runnables import RunnableLambda, RunnableParallel

final_chain = (
    prompt_template | llm_anthropic | str_parser | dictionary_maker_runnable | 
    RunnableParallel(branches = {"linkedin": chain_linkedIn, "instagram": insta_chain_runnable})
)

final_chain.invoke("Stranger Things")

#### Chain as a Runnable

In [ ]:
# TASK - 1 [Beautify Function]

def beautify(final_response: dict)-> dict:

    linkedin_response = final_response['branches']['linkedin']
    instagram_response = final_response['branches']['instagram']

    return {"linkedin": linkedin_response, "instagram": instagram_response}

beautify_runnable = RunnableLambda(beautify)

# TASK - 2 [Final Chain]

# Final Chain

# Beautified Chain
beautified_chain = final_chain | beautify_runnable
beautified_chain.invoke('Stranger Things')